# Litter detection and segmentation on UAVVaste

YOLOv8m trained with an extra NWD term in the box loss (helps with the many tiny objects), then SAM 2 turns each detected box into a mask. Evaluated on a random 80/20 split with pycocotools.

In [ ]:
!pip install -q ultralytics pycocotools

!git clone https://github.com/facebookresearch/segment-anything-2.git /kaggle/working/segment-anything-2
%cd /kaggle/working/segment-anything-2
!pip install -e . -q
!mkdir -p checkpoints
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt -O checkpoints/sam2_hiera_small.pt
%cd /kaggle/working

In [ ]:
import os
import json
import random
import shutil
from pathlib import Path

# the mount path depends on which copy of the dataset is attached
ann_path = next(Path("/kaggle/input").rglob("annotations.json"), None)
if ann_path is None:
    dataset_root = Path("/kaggle/input/datasets/uditanshsri/UAVVaste")
    ann_path = dataset_root / "annotations" / "annotations.json"
else:
    dataset_root = ann_path.parent.parent
print("annotations:", ann_path)

img_lookup = {p.name: p for p in dataset_root.rglob("*")
              if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}}
print(len(img_lookup), "images")

work_dir = Path("/kaggle/working/UAVVaste")
for split in ("train", "val"):
    (work_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (work_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

with open(ann_path) as f:
    coco = json.load(f)

images = {img["id"]: img for img in coco["images"]
          if Path(img["file_name"]).name in img_lookup}
all_ids = list(images)
random.seed(42)
random.shuffle(all_ids)
n_train = int(0.8 * len(all_ids))
train_ids = set(all_ids[:n_train])
val_ids = set(all_ids[n_train:])

anns_by_img = {i: [] for i in all_ids}
for ann in coco["annotations"]:
    if ann["image_id"] in anns_by_img:
        anns_by_img[ann["image_id"]].append(ann)

# COCO-format copy of the val split for pycocotools
val_coco = {
    "images": [images[i] for i in val_ids],
    "annotations": [a for i in val_ids for a in anns_by_img[i]],
    "categories": coco.get("categories", [{"id": 0, "name": "litter"}]),
}
with open(work_dir / "val_coco.json", "w") as f:
    json.dump(val_coco, f)

# YOLO labels: single class, normalised cx cy w h
for img_id, info in images.items():
    split = "train" if img_id in train_ids else "val"
    src = img_lookup[Path(info["file_name"]).name]
    dst = work_dir / "images" / split / src.name
    if not dst.exists():
        shutil.copy2(src, dst)

    W, H = info["width"], info["height"]
    lines = []
    for ann in anns_by_img[img_id]:
        x, y, w, h = ann["bbox"]
        if w <= 0 or h <= 0:
            continue
        lines.append(f"0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")
    (work_dir / "labels" / split / f"{src.stem}.txt").write_text("\n".join(lines))

yaml_path = work_dir / "uavvaste.yaml"
yaml_path.write_text(
    f"path: {work_dir}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: litter\n"
)
print(f"train: {len(train_ids)}  val: {len(val_ids)}")

In [ ]:
import inspect
import torch
from ultralytics import YOLO
from ultralytics.utils.loss import BboxLoss

# NWD constant. The NWD paper ties it to the dataset's average object size;
# 12 is close to the value they used for AI-TOD.
NWD_C = 12.0


def nwd(box1, box2, c=NWD_C, eps=1e-7):
    """Normalised Wasserstein distance between xyxy boxes, each modelled as a 2D Gaussian."""
    x1a, y1a, x2a, y2a = box1.chunk(4, -1)
    x1b, y1b, x2b, y2b = box2.chunk(4, -1)
    wa, ha = (x2a - x1a).clamp(min=eps), (y2a - y1a).clamp(min=eps)
    wb, hb = (x2b - x1b).clamp(min=eps), (y2b - y1b).clamp(min=eps)
    dx = (x1a + x2a - x1b - x2b) / 2
    dy = (y1a + y2a - y1b - y2b) / 2
    w2 = dx**2 + dy**2 + ((wa - wb)**2 + (ha - hb)**2) / 4
    return torch.exp(-w2.clamp(min=eps).sqrt() / c).squeeze(-1)


# Box loss becomes 0.5 * CIoU + 0.5 * NWD. Arguments are bound by name because
# BboxLoss.forward's signature has changed between ultralytics versions.
if not hasattr(BboxLoss, "forward_orig"):
    BboxLoss.forward_orig = BboxLoss.forward
_sig = inspect.signature(BboxLoss.forward_orig)


def forward_ciou_nwd(self, *args, **kwargs):
    loss_iou, loss_dfl = BboxLoss.forward_orig(self, *args, **kwargs)
    a = _sig.bind(self, *args, **kwargs).arguments
    fg = a["fg_mask"]
    if fg.sum() == 0:
        return loss_iou, loss_dfl
    weight = a["target_scores"].sum(-1)[fg]
    loss_nwd = ((1.0 - nwd(a["pred_bboxes"][fg], a["target_bboxes"][fg])) * weight).sum() / a["target_scores_sum"]
    return 0.5 * loss_iou + 0.5 * loss_nwd, loss_dfl


BboxLoss.forward = forward_ciou_nwd

model = YOLO("yolov8m.pt")  # yolov8s if 1024px doesn't fit in memory
model.train(
    data="/kaggle/working/UAVVaste/uavvaste.yaml",
    epochs=100,
    imgsz=1024,
    batch=8,
    device=0,
    workers=4,
    cache=True,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    project="/kaggle/working/runs",
    name="nwd_yolov8m",
)

In [ ]:
import sys
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("/kaggle/working/segment-anything-2")
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

weights = max(glob.glob("/kaggle/working/runs/nwd_yolov8m*/weights/best.pt"), key=os.path.getmtime)
print("weights:", weights)
detector = YOLO(weights)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sam2 = build_sam2("sam2_hiera_s.yaml",
                  "/kaggle/working/segment-anything-2/checkpoints/sam2_hiera_small.pt",
                  device=device)
predictor = SAM2ImagePredictor(sam2)


def detect_and_segment(image_path, conf=0.12, imgsz=1024):
    """YOLO boxes used as SAM 2 prompts. Returns image, boxes, masks (N, H, W) and % of the image covered."""
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    # low conf on purpose: SAM can't recover a missed object, but a loose box is fine
    boxes = detector(img, conf=conf, imgsz=imgsz, verbose=False)[0].boxes.xyxy.cpu().numpy()
    if len(boxes) == 0:
        return img, boxes, None, 0.0

    predictor.set_image(img)
    masks, _, _ = predictor.predict(box=boxes, multimask_output=False)
    if masks.ndim == 4:
        masks = masks.squeeze(1)
    masks = masks > 0.5
    coverage = masks.any(axis=0).mean() * 100
    return img, boxes, masks, coverage


val_images = sorted(Path("/kaggle/working/UAVVaste/images/val").glob("*.jpg"))
random.seed(42)
samples = random.sample(val_images, min(3, len(val_images)))

fig, axes = plt.subplots(len(samples), 3, figsize=(20, 6 * len(samples)))
fig.suptitle("NWD-YOLO box prompts + SAM 2 masks", fontsize=18, fontweight="bold", y=0.99)

for i, path in enumerate(samples):
    img, boxes, masks, coverage = detect_and_segment(path)

    boxed = img.copy()
    for x1, y1, x2, y2 in boxes.astype(int):
        cv2.rectangle(boxed, (x1, y1), (x2, y2), (255, 0, 0), 2)

    overlay = img.copy()
    for m in masks if masks is not None else []:
        overlay[m] = (overlay[m] * 0.5 + np.array([0, 120, 255]) * 0.5).astype(np.uint8)
        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)

    for ax, im in zip(axes[i], (img, boxed, overlay)):
        ax.imshow(im)
        ax.axis("off")
    if i == 0:
        axes[i, 0].set_title("Input", fontsize=14, fontweight="bold")
        axes[i, 1].set_title("YOLO boxes (conf=0.12)", fontsize=14, fontweight="bold")
    axes[i, 2].set_title(f"SAM 2 masks, litter covers {coverage:.2f}% of frame",
                         fontsize=12, fontweight="bold", color="darkblue")

plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.savefig("/kaggle/working/pipeline_examples.png", dpi=300)
plt.show()

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

coco_gt = COCO("/kaggle/working/UAVVaste/val_coco.json")
cat_id = coco_gt.getCatIds()[0] if coco_gt.getCatIds() else 0
name_to_id = {Path(img["file_name"]).name: img["id"] for img in coco_gt.dataset["images"]}

EVAL_CONF = 0.05  # keep low so COCOeval gets the full PR curve

box_preds, refined_preds = [], []
bbox_area_total = mask_area_total = 0.0

for n, path in enumerate(val_images, 1):
    if path.name not in name_to_id:
        continue
    image_id = name_to_id[path.name]
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)

    res = detector(img, conf=EVAL_CONF, imgsz=1024, verbose=False)[0].boxes
    boxes, scores = res.xyxy.cpu().numpy(), res.conf.cpu().numpy()
    if len(boxes) == 0:
        continue

    predictor.set_image(img)
    masks, _, _ = predictor.predict(box=boxes, multimask_output=False)
    if masks.ndim == 4:
        masks = masks.squeeze(1)

    for (x1, y1, x2, y2), score, mask in zip(boxes, scores, masks):
        box_preds.append({"image_id": image_id, "category_id": cat_id,
                          "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                          "score": float(score)})
        bbox_area_total += (x2 - x1) * (y2 - y1)
        mask_area_total += float(mask.sum())

        # replace the YOLO box with the tight box around the SAM mask
        ys, xs = np.nonzero(mask)
        if len(xs):
            refined_preds.append({"image_id": image_id, "category_id": cat_id,
                                  "bbox": [float(xs.min()), float(ys.min()),
                                           float(xs.max() - xs.min()), float(ys.max() - ys.min())],
                                  "score": float(score)})

    if n % 30 == 0 or n == len(val_images):
        print(f"{n}/{len(val_images)}")


def coco_bbox_eval(preds, out_path):
    with open(out_path, "w") as f:
        json.dump(preds, f)
    ev = COCOeval(coco_gt, coco_gt.loadRes(out_path), "bbox")
    ev.evaluate()
    ev.accumulate()
    ev.summarize()
    s = ev.stats * 100
    return [s[1], s[0], s[3], s[9]]  # AP50, AP50:95, AP_small, AR_small


print("\nYOLO boxes")
yolo_scores = coco_bbox_eval(box_preds, "/kaggle/working/preds_yolo.json")
print("\nYOLO + SAM 2 refined boxes")
sam_scores = coco_bbox_eval(refined_preds, "/kaggle/working/preds_yolo_sam2.json")
bg_removed = 100 * (bbox_area_total - mask_area_total) / bbox_area_total if bbox_area_total else 0.0

# numbers reported in the UAVVaste paper (MDPI Remote Sensing, 2021)
paper_scores = [51.2, 28.4, 14.8, 20.1]

metrics = ["mAP@50", "mAP@50:95", "AP small", "AR small"]
print(f"\n{'':<12}{'paper':>10}{'YOLO':>10}{'YOLO+SAM2':>12}")
for name, p, a, b in zip(metrics, paper_scores, yolo_scores, sam_scores):
    print(f"{name:<12}{p:>10.2f}{a:>10.2f}{b:>12.2f}")
print(f"\nbox area outside the SAM masks: {bg_removed:.1f}%")

x = np.arange(len(metrics))
w = 0.25
fig, ax = plt.subplots(figsize=(13, 6))
for k, (vals, label, color) in enumerate([
    (paper_scores, "UAVVaste paper", "#7f7f7f"),
    (yolo_scores, "NWD-YOLOv8m", "#1f77b4"),
    (sam_scores, "NWD-YOLOv8m + SAM 2", "#2ca02c"),
]):
    bars = ax.bar(x + (k - 1) * w, vals, w, label=label, color=color)
    ax.bar_label(bars, fmt="%.1f", fontsize=9)
ax.set_xticks(x, metrics)
ax.set_ylabel("%")
ax.set_title("UAVVaste validation split")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("/kaggle/working/baseline_comparison.png", dpi=300)
plt.show()